In [1]:
import pandas as pd
import json
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels, load_runs_from_path
import ir_measures
from src.data import parse_file_names
from pathlib import Path

from topic_gen import logger
logger.setLevel("DEBUG")

In [3]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):
    names.append(result)

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)
    metadata_records.append(metadata)

    # predictions
    qrels = binarize_qrels(ir_measures.read_trec_qrels(
        os.path.join(BASE_DIR, result, "qrels.csv.gz")))
    predictions.append(qrels)

In [4]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 12 / 2939
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 11 / 2940
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 4 / 2947
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 2 / 2949
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 15 / 2936
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 9 / 2942
[topic_gen] [INFO

In [5]:
# metadata table
metadata = pd.DataFrame(metadata_records)
metadata = metadata.join(pd.json_normalize(
    metadata["topics"]).add_prefix("topics_"))
metadata.drop(columns=["topics"], inplace=True)
metadata["topics_prompt"] = metadata["topics_prompt"].apply(
    lambda p: str(Path(p).stem) if pd.notnull(p) else "human")
metadata["prompt"] = metadata["prompt"].apply(lambda p: str(Path(p).stem))
metadata["model"] = metadata["model"].str.replace("-MT1000", "")
metadata["model"] = metadata["model"].str.replace("-MT100", "")

In [6]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


table = pd.DataFrame(res)
table["score"] = table.apply(format_score, axis=1)
table = table.pivot(index="name", columns="measure",
                    values="score").reset_index()

In [7]:
table = table.merge(metadata, left_on="name", right_on="date")

# Prerequisite
### Can we reproduce the results from Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [8]:
table[(table["prompt"] == "-DNA-zero-shot")
      & (table["topics_prompt"] == "human")][["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.02,0.11 ± 0.01,0.89 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.02
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
53,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
80,gpt-oss-120B,-DNA-zero-shot,0.69 ± 0.01,0.15 ± 0.01,0.84 ± 0.01


### Do LLM judges benefit from additional information in the topic?
Binary Setting:
- Für `GPT` scheint mehr Kontext schlechter zu sein. Für `Qwen` scheint mehr Kontext besser zu sein (mit ausnahmen...).
- Genereller ist der Effekt gering

In [9]:
partial_annotation_prompts = [
    "-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative",
    "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"]

table["prompt"] = pd.Categorical(table["prompt"], partial_annotation_prompts)
table[table["topics_prompt"] == "human"].sort_values(by=["model", "prompt"])[
    ["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
53,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
80,gpt-oss-120B,-DNA-zero-shot,0.69 ± 0.01,0.15 ± 0.01,0.84 ± 0.01
86,gpt-oss-120B,-DNA-zero-shot-masked-title,0.69 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
82,gpt-oss-120B,-DNA-zero-shot-masked-description,0.65 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
83,gpt-oss-120B,-DNA-zero-shot-masked-narrative,0.64 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
84,gpt-oss-120B,-DNA-zero-shot-masked-title-description,0.63 ± 0.02,0.18 ± 0.02,0.81 ± 0.01
85,gpt-oss-120B,-DNA-zero-shot-masked-title-narrative,0.69 ± 0.02,0.15 ± 0.01,0.83 ± 0.01
81,gpt-oss-120B,-DNA-zero-shot-masked-description-narrative,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.02
